# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library and the standard Croissant schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata info
meta = dataset.metadata
print(f"Dataset Title: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Let's discover the available record sets in the dataset schema.

All Croissant entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Explore available record sets, fields, and IDs
record_sets = dataset.metadata.get_record_sets()

for rs in record_sets:
    print(f"Record Set: {rs.name} (id: {rs.id})")
    for field in rs.get_fields():
        print(f"  Field: {field.name} (id: {field.id})  type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
    print('-'*50)

# For demonstration, pick the first available record set
main_record_set_id = record_sets[0].id if record_sets else None
print(f"Main record set id: {main_record_set_id}")

## 3. Data Extraction
Load records from the main record set into a pandas DataFrame. Each record set and field is referenced by its Croissant `@id`.

In [ ]:
dataframes = {}
if main_record_set_id is not None:
    # Load records of first record set
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df

    print(f"Columns in {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in schema.")

## 4. Exploratory Data Analysis (EDA)
We'll now carry out some basic EDA on the main record set, operating on numeric fields, filtering records, and grouping by demographic variables.

All field and group references are always by their Croissant `@id`.

In [ ]:
# Find numeric fields for analysis (e.g., GAD-7, PHQ-9, PSQ scores)
main_rs = [rs for rs in dataset.metadata.get_record_sets() if rs.id == main_record_set_id][0]
numeric_fields = [
    field.id for field in main_rs.get_fields()
    if hasattr(field, "data_type") and field.data_type in ["schema:Integer", "schema:Float", "Float", "Integer", "Number"]
]
print("Numeric fields (by @id):", numeric_fields)

# We'll select the first numeric field (e.g., GAD-7 total) for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field for filtering/analysis: {numeric_field_id}")
else:
    numeric_field_id = None

df = dataframes[main_record_set_id]

# Ensure the column exists and is numeric
if numeric_field_id is not None and numeric_field_id in df.columns:
    # Convert column to numeric if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-9)
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Choose a group field (e.g., demographic variable)
    # We'll prefer fields with a small number of unique categorical values (e.g., gender, marital status)
    candidate_group_fields = [
        field.id for field in main_rs.get_fields() if field.id in df.columns and df[field.id].nunique() < df.shape[0] // 3
    ]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id:
        print(f"Grouping by {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print("No suitable group field found for grouping.")
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution of a selected numeric variable (by field `@id`) and the grouping variable, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group_field_id if available
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded the Croissant schema and records from the FAIR² Kilifi Mental Health Survey dataset using `mlcroissant`.
- Explored record set, field, and column structure by their Croissant `@id`s.
- Extracted the main record set into a pandas DataFrame for analysis.
- Performed basic exploratory and group statistics on identified numeric variables (by `@id`).
- Visualized the data, including distributions and groupwise comparisons, to prepare for further domain-specific analyses.

This workflow demonstrates reproducible and schema-driven dataset exploration using the `mlcroissant` library.
